# 06 — Disease Classification & Pattern Validation

**Dataset:** GSE2034 — Breast cancer bone relapse prediction

**Label:** `characteristics_ch1` → bone relapse (1=yes / 0=no)

Classifiers: Logistic Regression, Random Forest, SVM  
Evaluation: stratified k-fold CV, ROC-AUC, permutation test

In [ ]:
# ── Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Disease-Gene-Pattern-Detection'
sys.path.insert(0, f'{PROJECT_ROOT}/src')
print('Project root:', PROJECT_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, permutation_test_score
)
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Libraries loaded.')


## Step 1 — Load Data & Extract Labels

In [ ]:
# ── Load metadata ──────────────────────────────────────────────────────────
meta = pd.read_csv(
    f'{PROJECT_ROOT}/data/processed/metadata_clean.csv', index_col=0
)

# ── Extract label from characteristics_ch1 ────────────────────────────────
# Raw value: "bone relapses (1=yes, 0=no): 0"  or  "bone relapses (1=yes, 0=no): 1"
# We split on ':' and take the last part → '0' or '1'
raw = meta['characteristics_ch1'].dropna()

label_series = (
    raw
    .str.split(':')
    .str[-1]
    .str.strip()
    .map({'0': 'No Relapse', '1': 'Bone Relapse'})
)

print('Label distribution:')
print(label_series.value_counts())
print(f'\nMapped  : {label_series.notna().sum()}')
print(f'Unmapped: {label_series.isna().sum()}  (will be excluded)')


In [ ]:
# ── Load PCA scores ────────────────────────────────────────────────────────
pca_scores = pd.read_csv(
    f'{PROJECT_ROOT}/data/processed/pca_scores.csv', index_col=0
)

valid_labels = label_series.dropna()
common       = pca_scores.index.intersection(valid_labels.index)

print(f'Samples in PCA scores   : {len(pca_scores)}')
print(f'Samples with valid label: {len(valid_labels)}')
print(f'Common (usable)         : {len(common)}')

if len(common) == 0:
    raise ValueError(
        'No overlapping samples found.\n'
        'Make sure sample IDs (GSM...) match in both pca_scores and metadata.'
    )


In [ ]:
# ── Encode labels ──────────────────────────────────────────────────────────
labels_raw = valid_labels.loc[common]

le = LabelEncoder()
y  = le.fit_transform(labels_raw)

print(f'Classes: {le.classes_}')
for cls, code in zip(le.classes_, np.unique(y)):
    print(f'  {cls} → {code}  (n={(y == code).sum()})')

is_binary = (len(le.classes_) == 2)

if len(np.unique(y)) < 2:
    raise ValueError('Only one class found — label extraction failed.')

# Auto CV folds: never exceed smallest class size
min_class_n = int(np.bincount(y).min())
N_SPLITS    = min(5, min_class_n)
print(f'\nUsing {N_SPLITS}-fold stratified CV'
      f' (smallest class = {min_class_n} samples)')


In [ ]:
# ── Feature matrix ─────────────────────────────────────────────────────────
N_PCS = min(20, pca_scores.shape[1])
X     = pca_scores.loc[common].iloc[:, :N_PCS].values

print(f'X : {X.shape}  (samples × PCs)')
print(f'y : {y.shape}')

# Class distribution pie
counts = pd.Series(labels_raw).value_counts()
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=['steelblue', 'crimson'], startangle=90)
ax.set_title('Class Distribution\n(GSE2034 — Bone Relapse)')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/06_class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()


## Step 2 — Cross-Validation

In [ ]:
scoring = 'roc_auc'

classifiers = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000, C=1.0, solver='lbfgs'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', probability=True, random_state=42
    ),
}

cv         = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
cv_results = {}

for name, clf in classifiers.items():
    fold_scores = cross_val_score(clf, X, y, cv=cv, scoring=scoring)
    cv_results[name] = {
        'mean': fold_scores.mean(),
        'std':  fold_scores.std(),
        'scores': fold_scores
    }
    print(f'{name:<25}  AUC = {fold_scores.mean():.4f} ± {fold_scores.std():.4f}')


In [ ]:
names = list(cv_results.keys())
means = [cv_results[n]['mean'] for n in names]
stds  = [cv_results[n]['std']  for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(names, means, xerr=stds,
               color='steelblue', alpha=0.8,
               error_kw={'ecolor': 'black', 'capsize': 5})
ax.axvline(0.5, ls='--', color='grey', label='Chance (0.5)')
ax.set_xlabel(f'ROC-AUC ({N_SPLITS}-fold CV)')
ax.set_title('Classifier Comparison — PCA Features\n(Bone Relapse vs No Relapse)')
ax.set_xlim(0, 1.1)
for bar, m, s in zip(bars, means, stds):
    ax.text(min(m + s + 0.02, 1.05),
            bar.get_y() + bar.get_height() / 2,
            f'{m:.3f}', va='center', fontsize=10)
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/06_cv_auc_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()


## Step 3 — ROC Curve

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(f'Train: {len(y_tr)}  |  Test: {len(y_te)}')
print(f'Test class counts → {dict(zip(le.classes_, np.bincount(y_te)))}')

best_clf = RandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1
)
best_clf.fit(X_tr, y_tr)

y_pred = best_clf.predict(X_te)
y_prob = best_clf.predict_proba(X_te)[:, 1]

auc = roc_auc_score(y_te, y_prob)
fpr, tpr, _ = roc_curve(y_te, y_prob)

print(f'\nTest AUC: {auc:.4f}')
print()
print(classification_report(y_te, y_pred, target_names=le.classes_))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color='crimson', lw=2,
             label=f'Random Forest (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8, label='Chance')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='crimson')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Bone Relapse Prediction')
axes[0].legend(loc='lower right')

# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_,
            annot_kws={'size': 14})
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix')

plt.suptitle('Random Forest — Test Set Performance', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/06_roc_confusion.png',
            dpi=150, bbox_inches='tight')
plt.show()


## Step 4 — Permutation Test

In [ ]:
clf_perm = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)

obs_score, perm_scores, p_value = permutation_test_score(
    clf_perm, X, y,
    scoring=scoring,
    cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42),
    n_permutations=200,
    random_state=42,
    n_jobs=-1
)

print(f'Observed AUC  : {obs_score:.4f}')
print(f'Null mean AUC : {perm_scores.mean():.4f}')
print(f'p-value       : {p_value:.4f}')
print(f'Significant   : {"YES ✓" if p_value < 0.05 else "NO ✗"}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_scores, bins=30, color='steelblue',
        edgecolor='white', alpha=0.8, label='Null distribution')
ax.axvline(obs_score, color='crimson', lw=2.5,
           label=f'Observed AUC = {obs_score:.3f}')
ax.axvline(0.5, color='grey', lw=1, ls='--', label='Chance')
ax.set_xlabel('AUC')
ax.set_ylabel('Count')
ax.set_title(f'Permutation Test  (p = {p_value:.4f})')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/06_permutation_test.png',
            dpi=150, bbox_inches='tight')
plt.show()


## Step 5 — PC Feature Importance

In [ ]:
best_clf.fit(X, y)

importances = pd.Series(
    best_clf.feature_importances_,
    index=[f'PC{i+1}' for i in range(X.shape[1])]
).sort_values(ascending=True)

threshold = importances.quantile(0.75)
colors = ['crimson' if v >= threshold else 'steelblue'
          for v in importances.values]

fig, ax = plt.subplots(figsize=(7, 6))
importances.plot.barh(color=colors, alpha=0.85, ax=ax)
ax.axvline(threshold, ls='--', color='crimson', lw=0.8)
ax.set_title('Random Forest PC Importance\n(red = top-25% discriminative PCs)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/06_feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()

top_pcs = importances.tail(5).index.tolist()[::-1]
print(f'Top 5 discriminative PCs: {top_pcs}')


In [ ]:
# ── PCA scatter coloured by bone relapse ───────────────────────────────────
pca_plot = pca_scores.loc[common]

fig, ax = plt.subplots(figsize=(8, 6))
for code, cls, color in zip(
    [0, 1], le.classes_, ['steelblue', 'crimson']
):
    idx = (y == code)
    ax.scatter(pca_plot.iloc[idx, 0], pca_plot.iloc[idx, 1],
               label=cls, s=50, alpha=0.75, color=color)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PCA — PC1 vs PC2\nColoured by Bone Relapse Status')
ax.legend(title='Label')
ax.axhline(0, lw=0.5, color='grey')
ax.axvline(0, lw=0.5, color='grey')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/06_pca_by_label.png',
            dpi=150, bbox_inches='tight')
plt.show()


## Step 6 — Save Results

In [ ]:
cv_summary = pd.DataFrame({
    n: {'mean_auc': r['mean'], 'std_auc': r['std']}
    for n, r in cv_results.items()
}).T
cv_summary.to_csv(f'{PROJECT_ROOT}/data/results/06_cv_summary.csv')

importances.sort_values(ascending=False).to_csv(
    f'{PROJECT_ROOT}/data/results/06_pc_importances.csv',
    header=['importance']
)

prob_df = pd.DataFrame({
    'true_label':   labels_raw.values,
    'true_encoded': y,
    'prob_bone_relapse': best_clf.predict_proba(X)[:, 1]
}, index=common)
prob_df.to_csv(f'{PROJECT_ROOT}/data/results/06_predicted_probabilities.csv')

print('Files saved to data/results/:')
for f in sorted(os.listdir(f'{PROJECT_ROOT}/data/results/')):
    if f.startswith('06'):
        print(f'  {f}')
print()
print('=== CV Summary ===')
print(cv_summary.round(4))
